# Setup

In [7]:
import json, os, re
from pathlib import Path
from typing import Literal

import anthropic
import jinja2
import yaml
from dotenv import load_dotenv
from pydantic import BaseModel

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

load_dotenv(REPO / ".env", override=True)

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "No ANTHROPIC_API_KEY in the environment. "
        "Copy .env.example to .env, set the key, then re-run this cell."
    )

MODEL = "claude-sonnet-5"

client = anthropic.Anthropic()

print(f"model: {MODEL}")

model: claude-sonnet-5


# Judging a submission


In [8]:
CRITERIA = [
    {
        "id": "novel_wording",
        "text": "The summary uses different words than the prompt.",
        "guidance": (
            "PASS if the summary restates the task in the student's own phrasing. "
            "FAIL if it is largely copied spans from the problem statement, even when reordered. "
            "Domain nouns that have no natural synonym (variable names, 'array', 'average') "
            "do not count as copying."
        ),
    },
    {
        "id": "concrete_types",
        "text": "Types must be specific and concrete.",
        "guidance": (
            "PASS if each input and output names a definite type a C programmer could declare -- "
            "'int', 'float', 'array of 10 ints', 'null-terminated string'. "
            "FAIL for vague placeholders such as 'a number', 'some values', 'the data', "
            "or a quantity given with no type at all."
        ),
    },
    {
        "id": "constraints_noted",
        "text": "Some constraints must be noted, or explicitly described as absent.",
        "guidance": (
            "PASS if the artifact states at least one real constraint (value ranges, input size, "
            "sign, precision, ordering) OR explicitly says the problem states no constraints. "
            "FAIL if constraints are simply never mentioned -- silence is not the same as "
            "declaring there are none."
        ),
    },
]

print(f"{len(CRITERIA)} criteria: {', '.join(c['id'] for c in CRITERIA)}")

3 criteria: novel_wording, concrete_types, constraints_noted


## 2. Verdict

In [9]:
class Verdict(BaseModel):
    criterion_id: str
    verdict: Literal["PASS", "FAIL"]
    evidence: str
    confidence: Literal["low", "medium", "high"]


class JudgeResult(BaseModel):
    verdicts: list[Verdict]


# This mirrors prompts/base/output_schema.json -- keep the two in step.
print(json.dumps(JudgeResult.model_json_schema(), indent=2)[:400], "...")

{
  "$defs": {
    "Verdict": {
      "properties": {
        "criterion_id": {
          "title": "Criterion Id",
          "type": "string"
        },
        "verdict": {
          "enum": [
            "PASS",
            "FAIL"
          ],
          "title": "Verdict",
          "type": "string"
        },
        "evidence": {
          "title": "Evidence",
          "type": "string"
       ...


## 3. Rendering the prompt

In [10]:
PHASE = yaml.safe_load((REPO / "prompts/phases/problem_definition.yaml").read_text())
PROBLEM = yaml.safe_load((REPO / "cases/problems/grade_average.yaml").read_text())

_jinja = jinja2.Environment(trim_blocks=True, lstrip_blocks=True, undefined=jinja2.Undefined)


def render_user_prompt(problem: dict, artifact: dict, criteria: list[dict], attempt: int = 1) -> str:
    """Render the phase template. `artifact` is the student's submission."""
    return _jinja.from_string(PHASE["user_template"]).render(
        problem=problem, artifact=artifact, criteria_to_judge=criteria, attempt=attempt
    )

good_artifact = {
    "summary": (
        "I need to work out the class's mean score. The program reads how many students "
        "there are, then reads each student's mark, adds them up and divides by the count."
    ),
    "inputs": "int n (the student count), then n floats, one score per student",
    "outputs": "a single float: the mean of the n scores, printed to two decimal places",
}

prompt = render_user_prompt(PROBLEM, good_artifact, CRITERIA)
print(prompt)

PROBLEM STATEMENT
A teacher wants to know the average score of the students in a class.

Read an integer n, the number of students in the class. Then read n scores, one per
student, each a real number. Compute and print the average of those n scores, rounded
to two decimal places.

You may assume there is at least one student, and that every score is between 0 and
100 inclusive.


PUBLIC TEST CASES
- input: "3\n90 80 70" output: "80.00"
- input: "1\n55.5" output: "55.50"
- input: "4\n100 0 100 0" output: "50.00"

STUDENT ARTIFACT

Summary:
I need to work out the class's mean score. The program reads how many students there are, then reads each student's mark, adds them up and divides by the count.

Inputs:
int n (the student count), then n floats, one score per student

Outputs:
a single float: the mean of the n scores, printed to two decimal places

CRITERIA TO JUDGE
Return exactly one verdict for each of the following 3 criteria, using the id in brackets.
- [novel_wording] The summar

## 4. The judge call

One call judges every criterion at once. `output_format=JudgeResult` is what turns the reply into
a validated object instead of text — `resp.parsed_output` comes back as a `JudgeResult`, already
type-checked.

Note what is *absent*: no `temperature`. Deliberation is bought with `effort` and adaptive
thinking instead, which lets the model reason before committing to a verdict.

In [11]:
def judge(problem: dict, artifact: dict, criteria: list[dict], attempt: int = 1) -> JudgeResult:
    """Judge one submission against every criterion in a single call."""
    try:
        resp = client.messages.parse(
            model=MODEL,
            max_tokens=PHASE["model"]["max_output_tokens"],
            system=PHASE["system_prompt"],
            messages=[
                {"role": "user", "content": render_user_prompt(problem, artifact, criteria, attempt)}
            ],
            thinking={"type": "adaptive"},
            output_config={"effort": PHASE["model"]["effort"]},
            output_format=JudgeResult,
        )
    except anthropic.AuthenticationError:
        raise RuntimeError(
            "No valid API key. Copy .env.example to .env and set ANTHROPIC_API_KEY."
        ) from None
    except anthropic.RateLimitError as e:
        retry_after = e.response.headers.get("retry-after", "60")
        raise RuntimeError(f"Rate limited; retry after {retry_after}s.") from None
    except anthropic.APIStatusError as e:
        raise RuntimeError(f"API error {e.status_code}: {e.message}") from None
    except anthropic.APIConnectionError:
        raise RuntimeError("Could not reach the API. Check your connection.") from None

    if resp.stop_reason == "max_tokens":
        raise RuntimeError(
            "Response hit max_tokens and the verdict list is truncated. "
            "Raise max_output_tokens in the phase YAML, or lower `effort`."
        )
    return resp.parsed_output


result = judge(PROBLEM, good_artifact, CRITERIA)

for v in result.verdicts:
    print(f"{v.verdict:4}  {v.criterion_id:18} ({v.confidence})")
    print(f"        evidence: {v.evidence!r}\n")

PASS  novel_wording      (medium)
        evidence: "I need to work out the class's mean score. The program reads how many students there are, then reads each student's mark, adds them up and divides by the count."

PASS  concrete_types     (high)
        evidence: 'int n (the student count), then n floats, one score per student'

FAIL  constraints_noted  (high)
        evidence: ''

